# Maze Solver Training - Modular Version

This notebook uses modular code from `src/` folder:
- `src/model.py`: ResNet + GPT2 model architecture
- `src/tokenizer.py`: Simple tokenizer for maze sequences
- `src/dataset.py`: Dataset and DataLoader utilities
- `src/train_utils.py`: Training and evaluation functions

In [ ]:
import sys
import json
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

# Add src to path
sys.path.append('./src')

# Import our modules
from model import ResNetGPT2PrefixModel
from tokenizer import SimpleTokenizer
from dataset import MazeDataset, collate_fn
from train_utils import train_model, test_model

print("✓ All modules imported successfully")

## 1. Setup Tokenizer

In [ ]:
print("=" * 60)
print("TOKENIZER SETUP")
print("=" * 60)

tokenizer = SimpleTokenizer()

print(f"Vocabulary: {tokenizer.vocab}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Special tokens: BOS={tokenizer.bos_token_id}, EOS={tokenizer.eos_token_id}, PAD={tokenizer.pad_token_id}")

## 2. Load Data

In [ ]:
print("\n" + "=" * 60)
print("LOADING DATA")
print("=" * 60)

# Load JSON data
with open('../data/train.json', 'r') as f:
    train_entries = json.load(f)

with open('../data/test.json', 'r') as f:
    test_entries = json.load(f)

print(f"Training set: {len(train_entries)} examples")
print(f"Test set: {len(test_entries)} examples")

# Verify sample
print("\n" + "=" * 60)
print("VERIFICATION - Sample Entry")
print("=" * 60)
sample = train_entries[0]
print(f"Sample sequence: {sample['sequence']}")
token_ids = tokenizer.encode(sample['sequence'])
print(f"Token IDs: {token_ids}")
print(f"Decoded: '{tokenizer.decode_to_string(token_ids)}'")

## 3. Create Datasets and DataLoaders

In [ ]:
# Image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = MazeDataset(train_entries, tokenizer, transform)
test_dataset = MazeDataset(test_entries, tokenizer, transform)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id)
)

print(f"✓ Train loader: {len(train_loader)} batches")
print(f"✓ Test loader: {len(test_loader)} batches")

## 4. Initialize Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Model configuration
model = ResNetGPT2PrefixModel(
    vocab_size=len(tokenizer),
    hidden_size=128,           # Embedding dimension
    num_layers=2,              # GPT2 layers
    num_attention_heads=2,     # Attention heads
    num_prefix_tokens=8,       # Number of prefix tokens
    dropout=0.4,               # Dropout rate
    resnet_frozen=True         # Keep ResNet frozen initially
)

model = model.to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Prefix tokens: {model.num_prefix_tokens}")

## 5. Train Model

In [ ]:
print("\n" + "=" * 60)
print("TRAINING PHASE")
print("=" * 60)
print(f"Training on {len(train_entries)} mazes for 75 epochs...")
print("=" * 60)

final_loss = train_model(model, train_loader, epochs=75, lr=5e-4, device=device)

print(f"\nTraining completed. Final loss: {final_loss:.6f}")

## 6. Evaluate on Training Set

In [ ]:
print("\n" + "=" * 60)
print("TRAINING SET EVALUATION")
print("=" * 60)
print(f"Evaluating model performance on training set ({len(train_entries)} mazes)...")
print("=" * 60)

train_correct = test_model(model, train_loader, device, tokenizer, num_samples=len(train_entries))

print("=" * 60)
print(f"Training Accuracy: {train_correct}/{len(train_entries)} ({100*train_correct/len(train_entries):.1f}%)")
print("=" * 60)

## 7. Evaluate on Test Set

In [ ]:
print("\n" + "=" * 60)
print("TEST SET EVALUATION (UNSEEN DATA)")
print("=" * 60)
print(f"Evaluating on held-out test set ({len(test_entries)} mazes)...")
print("=" * 60)

test_correct = test_model(model, test_loader, device, tokenizer, num_samples=len(test_entries))

print("=" * 60)
print(f"Test Accuracy: {test_correct}/{len(test_entries)} ({100*test_correct/len(test_entries):.1f}%)")
print("=" * 60)

## 8. Final Results & Save Model

In [ ]:
print("\n" + "=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)

train_acc_pct = 100 * train_correct / len(train_entries)
test_acc_pct = 100 * test_correct / len(test_entries)
gen_gap = train_acc_pct - test_acc_pct

print(f"Final Training Loss:    {final_loss:.6f}")
print(f"Training Accuracy:      {train_correct}/{len(train_entries)} ({train_acc_pct:.1f}%)")
print(f"Test Accuracy:          {test_correct}/{len(test_entries)} ({test_acc_pct:.1f}%)")
print(f"Generalization Gap:     {gen_gap:.1f}%")
print("=" * 60)

# Performance assessment
if final_loss < 0.1 and test_acc_pct >= 80:
    print("\n🎉 SUCCESS! Model generalizes well to unseen mazes!")
elif final_loss < 0.2 and test_acc_pct >= 60:
    print("\n✓ Good progress! Model is learning but could improve")
else:
    print("\n⚠️  Model may need more training or hyperparameter tuning")
    if gen_gap > 50:
        print("   → High generalization gap suggests overfitting")
    elif train_acc_pct < 50:
        print("   → Low training accuracy suggests underfitting or insufficient capacity")

# Save model
checkpoint_path = "../models/resnet_gpt2_prefix.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': len(tokenizer),
    'hidden_size': model.hidden_size,
    'num_prefix_tokens': model.num_prefix_tokens,
    'final_loss': final_loss,
    'train_accuracy': train_acc_pct,
    'test_accuracy': test_acc_pct,
    'generalization_gap': gen_gap,
}, checkpoint_path)

print(f"\n💾 Model checkpoint saved to {checkpoint_path}")
print("=" * 60)